## Clustering of embeddings and cluster analysis

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import pickle
import os
import math
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
from sklearn.metrics import pairwise_distances, silhouette_score, v_measure_score, calinski_harabasz_score, adjusted_rand_score, davies_bouldin_score
from sklearn.cluster import KMeans, AgglomerativeClustering, HDBSCAN
from sklearn.manifold import TSNE
import umap
from PIL import Image
from scipy.stats import entropy, zscore
from utils.mypath import MyPath
from utils.mypalette import mutation_palette, condition_palette, day_palette, cell_line_palette
from utils.snn import *
from utils.start_notebook import setup_experiment, load_process_handcrafted_features
from sklearnex import patch_sklearn
patch_sklearn()
  
warnings.filterwarnings('ignore')


x = '240813_5'
single_exp = ['E2']
cluster_0 = False
emb_type = 'deep'
assert emb_type in ['deep', 'handcrafted']
saving_dir = f'../results/{x}/clustering/{("_").join(single_exp)}'
os.makedirs(saving_dir, exist_ok=True)


if emb_type == 'deep':
    if os.path.exists(os.path.join('../results', x, 'clustering', f'best_clustering_labels_{("_").join(single_exp)}.csv')):
        best_labels_path = os.path.join('../results', x, 'clustering', f'best_clustering_labels_{("_").join(single_exp)}.csv')
    else:
        best_labels_path = None

annotations_df, raw_emb = setup_experiment(x, single_exp)
if emb_type == 'handcrafted':
    hand_crafted_annotations, hc_features = load_process_handcrafted_features(annotations_df)
    emb_start = hc_features
else:
    emb_start = raw_emb
    cluster_evaluation_path = f'../results/{x}/clustering/clustering_results_{("_").join(single_exp)}.csv'

cluster_evaluations = pd.read_csv(cluster_evaluation_path)
categorical_labels = annotations_df['mut_cond'].values

embeddings = {}
# Compute UMAP embeddings with a range of n_neighbors parameter
#n_neighbors_range = [15, 50, 100, 150, 200, 300, 600]
n_neighbors_range = [15]
for n_neighbors in n_neighbors_range:
    umap_emb = umap.UMAP(n_neighbors=n_neighbors, random_state=0).fit_transform(emb_start)
    embeddings[f'umap_n_neighbors_{n_neighbors}'] = umap_emb

# Specify the desired order for mut_cond
desired_order = ['CTRL_untreated', 'CTRL_A1 low', 'CTRL_A1 high',
                 'VCP_untreated', 'VCP_A1 low', 'VCP_A1 high'][::-1]
# Remove classes from desired_order that are not present in the mut_cond column
desired_order = [cls for cls in desired_order if cls in annotations_df['mut_cond'].unique()]
# Convert mut_cond to a categorical type with the specified order
annotations_df['mut_cond'] = pd.Categorical(annotations_df['mut_cond'], categories=desired_order, ordered=True)
annotations_df['mut_cond'] = annotations_df['mut_cond'].apply(lambda x: x.replace('_', '\n'))

dataset_size = len(annotations_df)
cluster_evaluations = cluster_evaluations[cluster_evaluations['unassigned'] <= 0.15 * dataset_size]

# Plot evaluation metrics

In [ ]:
xticks = cluster_evaluations['embedding'].unique()
xticks = [x.replace('_', '\n', 1) for x in xticks]
xticks = [x.replace('n_neighbors_', '', 1) for x in xticks]

cluster_evaluations = cluster_evaluations[cluster_evaluations['embedding']!= 'phate']

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
sns.boxplot(x='embedding', y='calinski_harabasz', hue='method', data=cluster_evaluations, palette='Set2', ax=ax[0])
ax[0].set_title('C-H by Embedding and Clustering Method')
ax[0].set_ylabel('C-H Score')

sns.boxplot(x='embedding', y='silhouette', hue='method', data=cluster_evaluations, palette='Set2', ax=ax[1])
ax[1].set_title('Silhouette Scores by Embedding and Clustering Method')
ax[1].set_ylabel('Silhouette Score')
plt.suptitle('Label agnostic evaluation metrics')

for i in range(2):
    ax[i].set_xticklabels(xticks)
    ax[i].grid(axis='y')
    ax[i].set_xlabel('Embedding')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
sns.boxplot(x='embedding', y='vmeasure', hue='method', data=cluster_evaluations, palette='Set2', ax=ax[0])
ax[0].set_title('V-measure Scores by Embedding and Clustering Method')
ax[0].set_ylabel('V-measure Score')

sns.boxplot(x='embedding', y='ARI', hue='method', data=cluster_evaluations, palette='Set2', ax=ax[1])
ax[1].set_title('ARI Scores by Embedding and Clustering Method')
ax[1].set_ylabel('Silhouette Score')
plt.suptitle('Label aware evaluation metrics')

for i in range(2):
    ax[i].set_xticklabels(xticks)
    ax[i].grid(axis='y')
    ax[i].set_xlabel('Embedding')
plt.show()

In [ ]:
# Plot ARI vs vmeasure scores
plt.figure(figsize=(6, 8))
sns.scatterplot(x='ARI', y='vmeasure', data=cluster_evaluations)
plt.title('ARI vs V-measure Scores')
plt.xlabel('ARI Score')
plt.ylabel('V-measure Score')
plt.grid(True)
plt.show()

# Compute correlation
correlation = cluster_evaluations['ARI'].corr(cluster_evaluations['vmeasure'])
print(f'Correlation between ARI and V-measure scores: {correlation}')

In [ ]:
scores = ['ARI', 'vmeasure', 'silhouette', 'calinski_harabasz']
for sc in scores:
    cluster_evaluations[sc+'_zscore'] = zscore(cluster_evaluations[sc])

cluster_evaluations = cluster_evaluations.sort_values(by='vmeasure_zscore', ascending=False)
#cluster_evaluations.head(10)

In [ ]:
compute_weighted_score = True

if compute_weighted_score:
    weight = 0.7
    
    ARI_used = 1
    VMeasure_used = 1
    Calinski_used = 1
    Silhouette_used = 1
    
    label_aware_weight = 0.5 if ARI_used and VMeasure_used else 1
    label_agnostic_weight = 0.5 if Calinski_used and Silhouette_used else 1
    
    cluster_evaluations['weighted_score'] = weight * label_aware_weight *(ARI_used*cluster_evaluations['ARI_zscore'] + VMeasure_used*cluster_evaluations['vmeasure_zscore']) + \
                                            (1-weight) * label_agnostic_weight *(Calinski_used*cluster_evaluations['calinski_harabasz_zscore']+Silhouette_used*cluster_evaluations['silhouette_zscore'])
    
    cluster_evaluations = cluster_evaluations.sort_values(by='weighted_score', ascending=False)
    
cluster_evaluations.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

order=list(cluster_evaluations['embedding'].unique()[:-2]) + ['umap_n_neighbors_600', 'phate']

sns.boxplot(x='embedding', y='weighted_score', hue='method', data=cluster_evaluations, palette='Set2',
order=order)
plt.title(f'Weighted Score by Embedding and Clustering Method - {single_exp[0]}')
plt.xticks(ticks=range(len(order)), labels=[label.replace('_n_neighbors_', '\nn neighbors=') for label in order])
plt.xlabel('Embedding')
plt.ylabel('Weighted Score')
plt.savefig(os.path.join(saving_dir, f'weighted_score_{single_exp[0]}.pdf'))
plt.show()

# Compute/retrieve best clustering labels

In [ ]:
from sklearn.neighbors import NearestNeighbors
emb_to_analyze = embeddings['umap_n_neighbors_15']

legend_order = ['CTRL1', 'CTRL2', 'CTRL4', 'CTRL5', 'GLIA', 'GLIB', 'CBI', 'CBID', 'CBIE']

if best_labels_path is not None:
    if ((os.path.exists(best_labels_path)) and (emb_type == 'deep')):
        best_labels_df = pd.read_csv(best_labels_path)
        best_labels = best_labels_df['cluster']

else:

    cl_method = SNN(
                neighbor_num=200,
                min_shared_neighbor_proportion=0.9,
                clustering_algo='DBSCAN'
    )

    best_labels = cl_method.fit_predict(emb_to_analyze)
    annotations_df['cluster'] = best_labels

    assigned_indices = best_labels != -1
    unassigned_indices = np.where(best_labels == -1)[0]

    # PLOT EMBEDDINGS COLORED BY MUTATION, CONDITION AND CLUSTER LABELS
    fig, axes = plt.subplots(1, 4, figsize=(32, 8))
    for i, (label, palette_) in enumerate(zip(['cell_line', 'mutation', 'condition', 'cluster'], [cell_line_palette, mutation_palette, condition_palette, None])):
        hue_data = annotations_df[label] if label != 'cluster' else best_labels
        print(f"Plotting {label} with hue data shape: {hue_data.shape} and type: {type(hue_data)}")
        if label == 'cluster':
            sns.scatterplot(x=emb_to_analyze[assigned_indices, 0],
                            y=emb_to_analyze[assigned_indices, 1],
                            hue=best_labels[assigned_indices],
                            palette='tab20', ax=axes[i],
                            s=12, alpha=0.5)
            sns.scatterplot(x=emb_to_analyze[~assigned_indices, 0],
                            y=emb_to_analyze[~assigned_indices, 1],
                            color='black', ax=axes[i],
                            s=12, alpha=0.5, label='Unassigned')
        else:
            sns.scatterplot(x=emb_to_analyze[:, 0],
                            y=emb_to_analyze[:, 1],
                            hue=hue_data,
                            palette=palette_, ax=axes[i],
                            s=12, alpha=0.5)

        axes[i].set_title(label)
        if label == 'cell_line':
            handles, labels = axes[i].get_legend_handles_labels()
            ordered_handles_labels = sorted(zip(handles, labels), key=lambda x: legend_order.index(x[1]) if x[1] in legend_order else float('inf'))
            ordered_handles, ordered_labels = zip(*ordered_handles_labels)
            axes[i].legend(ordered_handles, ordered_labels, markerscale=2, fancybox=True, loc='best')
    plt.show()

    if len(unassigned_indices) > 0:
        # Find the 25 nearest assigned neighbors for each unassigned sample
        nbrs = NearestNeighbors(n_neighbors=25).fit(emb_to_analyze[assigned_indices])
        distances, indices = nbrs.kneighbors(emb_to_analyze[unassigned_indices])
        
        # Assign labels to unassigned samples based on majority vote of nearest neighbors
        lab_corrected = best_labels.copy()
        for idx, neighbors in zip(unassigned_indices, indices):
            neighbor_labels = best_labels[assigned_indices][neighbors]
            majority_label = np.bincount(neighbor_labels).argmax()
            lab_corrected[idx] = majority_label
        
        # Compute silhouette score
        silhouette_avg = silhouette_score(emb_to_analyze, lab_corrected)
        sil_previous = silhouette_score(emb_to_analyze, best_labels)
        print("Silhouette Score was: ", sil_previous, " and now is: ", silhouette_avg)
        
        # Compute Calinski-Harabasz index
        calinski_harabasz_avg = calinski_harabasz_score(emb_to_analyze, lab_corrected)
        cal_previous = calinski_harabasz_score(emb_to_analyze, best_labels)
        print("Calinski-Harabasz Index was: ", calinski_harabasz_avg, " and now is: ", calinski_harabasz_avg)
        
        # Compute V-measure
        v_measure = v_measure_score(categorical_labels, lab_corrected)
        vm_previous = v_measure_score(categorical_labels, best_labels)
        print("V-measure was: ", vm_previous, " and now is: ", v_measure)
        
        # Compute ARI
        adjusted_rand = adjusted_rand_score(categorical_labels, lab_corrected)
        ari_previous = adjusted_rand_score(categorical_labels, best_labels)
        print("ARI was: ", ari_previous, " and now is: ", adjusted_rand)
        
        best_labels = lab_corrected
    
if not cluster_0 and min(best_labels) == 0:
    best_labels = best_labels +1
annotations_df['cluster'] = best_labels

annotations_df.to_csv(os.path.join(saving_dir, f'best_clustering_labels_{("_").join(single_exp)}.csv'), index=False)

num_clusters = len(np.unique(best_labels))
if num_clusters > 20:
    cl_palette = sns.color_palette('tab20', 20) + sns.color_palette('husl', num_clusters - 20)
else:
    cl_palette = sns.color_palette('tab20', num_clusters)

fig, axes = plt.subplots(1, 4, figsize=(32, 8))
for i, (label, palette) in enumerate(zip(['cell_line', 'mutation', 'condition', 'cluster'], [cell_line_palette, mutation_palette, condition_palette, cl_palette])):
    hue_data = annotations_df[label] if label != 'cluster' else best_labels
    sns.scatterplot(x=emb_to_analyze[:, 0],
                    y=emb_to_analyze[:, 1],
                    hue=hue_data,
                    palette=palette, ax=axes[i],
                    s=12, alpha=0.5)
        
    axes[i].set_title(label)
    axes[i].legend(markerscale=2, fancybox=True, loc='best')
    if label == 'cell_line':
        handles, labels = axes[i].get_legend_handles_labels()
        ordered_handles_labels = sorted(zip(handles, labels), key=lambda x: legend_order.index(x[1]) if x[1] in legend_order else float('inf'))
        ordered_handles, ordered_labels = zip(*ordered_handles_labels)
    for lh in axes[i].get_legend().legendHandles:
        lh.set_alpha(1)
#plt.savefig(os.path.join(saving_dir, f'umap_n_neighbors_15_clustered_{("_").join(single_exp)}.pdf'))
plt.show()

In [ ]:
# Plot cluster label distribution as a histogram for each cell line and condition, ordered by mut_cond
mut_cond_order = ['CTRL\nuntreated', 'CTRL\nA1 low', 'CTRL\nA1 high',
                  'VCP\nuntreated', 'VCP\nA1 low', 'VCP\nA1 high']
mut_cond_order = [mc for mc in mut_cond_order if mc in annotations_df['mut_cond'].unique()]

# For each mut_cond, plot all cell lines in that mut_cond in the same row
cell_lines_by_mut_cond = []
for mc in mut_cond_order:
    cell_lines_in_mc = annotations_df[annotations_df['mut_cond'] == mc]['cell_line'].unique()
    cell_lines_by_mut_cond.append(list(cell_lines_in_mc))

nrows = len(mut_cond_order)
ncols = max(len(clist) for clist in cell_lines_by_mut_cond)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows), sharey=True)

# axes could be 1D if nrows==1, ensure always 2D
if nrows == 1:
    axes = np.expand_dims(axes, axis=0)
if ncols == 1:
    axes = np.expand_dims(axes, axis=1)

for row_idx, (mc, clist) in enumerate(zip(mut_cond_order, cell_lines_by_mut_cond)):
    for col_idx, cell_line in enumerate(clist):
        data = annotations_df[(annotations_df['cell_line'] == cell_line) & (annotations_df['mut_cond'] == mc)]
        axes[row_idx, col_idx].hist(
            data['cluster'],
            bins=np.arange(annotations_df['cluster'].min(), annotations_df['cluster'].max() + 2) - 0.5,
            color=cell_line_palette.get(cell_line, None)
        )
        mut = data["mutation"].iloc[0] if not data.empty else ""
        cond = data["condition"].iloc[0] if not data.empty else ""
        axes[row_idx, col_idx].set_title(f'{cell_line}\n{mc}\n{mut} {cond}\nN={len(data)}', fontsize=11)
        axes[row_idx, col_idx].set_xlabel('Cluster Label')
        axes[row_idx, col_idx].set_ylabel('Count')
        axes[row_idx, col_idx].set_xticks(np.arange(annotations_df['cluster'].min(), annotations_df['cluster'].max() + 1))
        axes[row_idx, col_idx].tick_params(axis='x')
    # Hide unused columns in this row
    for col_idx in range(len(clist), ncols):
        axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.show()


## Hand-crafted features mean and variance

In [ ]:
if emb_type == 'handcrafted':

    features_dict = {
        "Chebyshev Fourier Statistics": [
            f"chebyshev_fourier_stats-num_coeffs_400-num_bins_32-hist_bin{i}" for i in range(32)
        ],
        "Chebyshev Statistics": [
            f"chebyshev_stats-num_coeffs_400-num_bins_32-hist_bin{i}" for i in range(32)
        ],
        "Edge Statistics": [
            "edge_stats-direction_hist_bin0", "edge_stats-direction_hist_bin1",
            "edge_stats-direction_hist_bin2", "edge_stats-direction_hist_bin3",
            "edge_stats-direction_hist_bin4", "edge_stats-direction_hist_bin5",
            "edge_stats-direction_hist_bin6", "edge_stats-direction_hist_bin7",
            "edge_stats-direction_hist_diff_bin0", "edge_stats-direction_hist_diff_bin1",
            "edge_stats-direction_hist_diff_bin2", "edge_stats-direction_hist_diff_bin3",
            "edge_stats-direction_homogeneity", "edge_stats-direction_mean",
            "edge_stats-direction_median", "edge_stats-direction_variance",
            "edge_stats-edge_density", "edge_stats-magnitude_hist_bin0",
            "edge_stats-magnitude_hist_bin1", "edge_stats-magnitude_hist_bin2",
            "edge_stats-magnitude_hist_bin3", "edge_stats-magnitude_hist_bin4",
            "edge_stats-magnitude_hist_bin5", "edge_stats-magnitude_hist_bin6",
            "edge_stats-magnitude_hist_bin7", "edge_stats-magnitude_mean",
            "edge_stats-magnitude_median", "edge_stats-magnitude_variance"
        ],
        "Gabor Filter Features": [
            f"gabor-frequency_{freq}-theta_{theta}-sigma_x_{sigma_x}-sigma_y_{sigma_y}-{stat}"
            for freq in [0.05, 0.25]
            for theta in [0.0, 0.7853981633974483, 1.5707963267948966, 2.356194490192345]
            for sigma_x, sigma_y in [(1.0, 1.0), (3.0, 3.0)]
            for stat in ["mean_imag", "mean_real", "var"]
        ],
        "Haralick Texture Features": [
            f"haralick-distances_{dist}-angles_{angles}-{stat}"
            for dist in [1, 2]
            for angles in [
                "0.0_0.39269908169872414_0.7853981633974483_1.1780972450961724_1.5707963267948966_1.9634954084936207_2.356194490192345_2.748893571891069",
                "0.0_0.7853981633974483_1.5707963267948966_2.356194490192345"
            ]
            for stat in [
                "contrast", "correlation", "dissimilarity", "energy", "entropy", "homogeneity"
            ]
        ],
        "Image Statistics": [
            'image_stats-max', 'image_stats-mean',
            'image_stats-mean_intensity_above_threshold',
            'image_stats-num_pixels_above_threshold', 'image_stats-percent_maximal',
            'image_stats-percent_minimal', 'image_stats-percent_pixels_above_threshold',
            'image_stats-total_intensity', 'image_stats-variance_above_threshold',
            'image_stats-variance',
        ],
        "Intensity Statistics": [
            f"intensity_stats-hist_bin{i}" for i in range(32)
        ] + [
            "intensity_stats-hist_diff_bin0", "intensity_stats-hist_diff_bin1",
            "intensity_stats-hist_diff_bin2", "intensity_stats-hist_diff_bin3",
            "intensity_stats-hist_diff_bin4", "intensity_stats-hist_diff_bin5",
            "intensity_stats-hist_diff_bin6", "intensity_stats-hist_diff_bin7",
            "intensity_stats-hist_homogeneity", "intensity_stats-mean",
            "intensity_stats-median", "intensity_stats-variance"
        ],
        "Statistical Moments": [
            "moments-kurtosis", "moments-mean", "moments-skewness", "moments-variance"
        ],
        "Shannon Entropy": [
            "shannon_entropy-blue", "shannon_entropy-green", "shannon_entropy-red"
        ],
        "Tamura Features": [
            f"tamura-max_k_{k}-num_bins_16-{stat}"
            for k in [2, 3, 4]
            for stat in [
                "coarseness", "contrast", "directionality", 
                "hist_bin0", "hist_bin1", "hist_bin2"
            ]
        ]
    }

    # Compute the correlation between each feature and the UMAP coordinates
    correlations = {}
    for idx, feature in enumerate(hc_features.T):
        corr_x = np.corrcoef(hc_features[:, idx], embeddings['umap_n_neighbors_15'][:, 0])[0, 1]
        corr_y = np.corrcoef(hc_features[:, idx], embeddings['umap_n_neighbors_15'][:, 1])[0, 1]
        feature_name = hand_crafted_annotations.columns[idx + 1]  # +1 to account for the 'img_code' column
        correlations[idx] = {'feature': feature_name, 'corr_x': corr_x, 'corr_y': corr_y}

    # Convert the correlations dictionary to a DataFrame
    correlations_df = pd.DataFrame.from_dict(correlations, orient='index')
    correlations_df.columns = ['feature', 'corr_x', 'corr_y']

    high_corr_features = correlations_df[(abs(correlations_df['corr_x']) > 0.80) | (abs(correlations_df['corr_y']) > 0.80)]
    unique_high_corr_features = pd.DataFrame(columns=['feature', 'corr_x', 'corr_y'])

    for k, v in features_dict.items():
        high_corr_temp = high_corr_features[high_corr_features['feature'].isin(v)]
        if high_corr_temp.shape[0] != 0:
            highest_corr_x = high_corr_temp.loc[high_corr_temp['corr_x'].idxmax()]
            highest_corr_y = high_corr_temp.loc[high_corr_temp['corr_y'].idxmax()]
            unique_high_corr_features.loc[len(unique_high_corr_features)] = highest_corr_x
            unique_high_corr_features.loc[len(unique_high_corr_features)] = highest_corr_y
    
    unique_high_corr_features = unique_high_corr_features.drop_duplicates()

    for f in unique_high_corr_features['feature']:
        annotations_df[f] = hand_crafted_annotations[f]
        fig, axes = plt.subplots(1, 1, figsize=(8, 8))
        sns.scatterplot(x=emb_to_analyze[:, 0],
                        y=emb_to_analyze[:, 1],
                        hue=annotations_df[f],
                        ax=axes,
                        s=14, alpha=0.7)
            
        axes.set_title(f)
        axes.legend(markerscale=2, fancybox=True, loc='best')
        for lh in axes.get_legend().legendHandles:
            lh.set_alpha(1)
    plt.show()

## Radar plots

In [ ]:
import math

def make_spider(df, row, color, labels, subplot_index, total_subplots, cluster_0):
    categories = list(df)[0:]
    N = len(categories)

    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]

    n_cols = 4 if N>6 else 3

    plt.subplot(total_subplots // n_cols + (total_subplots % n_cols > 0), n_cols, subplot_index + 1, polar=True)

    plt.xticks(angles[:-1], categories, color='black', size=8)
    plt.tick_params(axis='x', pad=10)

    plt.yticks([0.25, 0.5, 0.75], color="black", size=7)
    plt.ylim(0, 1.0)

    values = df.reset_index().loc[row].values.tolist()[1:]
    values += values[:1]
    linewidht = 3.5 if total_subplots > 1 else 2
    plt.plot(angles, values, color=color, linewidth=3.5, linestyle='solid', label=labels[row])
    plt.fill(angles, values, color=color, alpha=0.6 if total_subplots > 1 else 0.3)

    #title_fontweight = 'bold' if total_subplots > 1 else 'normal'
    plt.title(labels[row], size=11, color=color, y=1.2, fontweight='bold')

# Calculate the proportion of classes in each cluster
class_proportion = annotations_df.groupby(['cluster', 'mut_cond']).size().unstack(fill_value=0)
class_proportion = class_proportion.div(class_proportion.sum(axis=1), axis=0)
adjustment = 0 if cluster_0 else 1

# Define labels for the legend
labels = [f'Cluster {i+adjustment}' for i in range(len(class_proportion))]

# Plot spider charts for each cluster
total_clusters = len(class_proportion)
plt.figure(figsize=(18, 18))

for i, cluster_label in enumerate(class_proportion.index):
    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
    make_spider(class_proportion, adjusted_cluster_label, color=palette[adjusted_cluster_label], labels=labels, subplot_index=i, total_subplots=total_clusters, cluster_0=cluster_0)
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'spider_charts_clusters_{("_").join(single_exp)}.pdf'))
plt.show()

# Plot spider charts for each cluster in separate figures
for i, cluster_label in enumerate(class_proportion.index):
    plt.figure(figsize=(10, 10))
    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
    make_spider(class_proportion, adjusted_cluster_label, color=palette[adjusted_cluster_label], labels=labels, subplot_index=0, total_subplots=1, cluster_0=cluster_0)
    plt.savefig(os.path.join(saving_dir, f'spider_chart_single_{adjusted_cluster_label}.pdf'))
    plt.show()

In [ ]:
# Function to create a spider plot for each category
def make_spider_for_category(df, category, clusters, color, subplot_index, total_subplots):
    categories = list(df.columns)
    N = len(categories)
    global_max = df.max().max()

    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]

    # Calculate number of columns for subplot layout
    n_cols = 3 if total_subplots > 4 else 2
    n_rows = (total_subplots // n_cols) + (total_subplots % n_cols > 0)
    
    ax = plt.subplot(n_rows, n_cols, subplot_index + 1, polar=True)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, color='black', size=8)
    ax.tick_params(axis='x', pad=10)
    
    ax.set_ylim(0, global_max)
    #ax.yticks(color="black", size=7)

    # Plot values for the category
    values = df.loc[category].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2, linestyle='solid', label=category)
    ax.fill(angles, values, color=color, alpha=0.4)

    ax.set_title(f'\n\n\n\n{category}\n\n', size=15, color=color, y=1.05, fontweight='bold')

# Calculate the proportion of clusters for each category
category_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
category_proportion = category_proportion.div(category_proportion.sum(axis=1), axis=0)

# Define the desired order for categories
desired_order = ['CTRL\nuntreated', 'CTRL\nA1 low', 'CTRL\nA1 high',
                 'VCP\nuntreated', 'VCP\nA1 low', 'VCP\nA1 high']
desired_order = [category for category in desired_order if category in category_proportion.index]
print("Categories in category_proportion:")
print(category_proportion.index)

if 'VCP\nuntreated' in category_proportion.index:
    print("Proportions for VCP\nuntreated:")
    print(category_proportion.loc['VCP\nuntreated'])

# Define colors for the categories based on condition_palette
category_colors = [condition_palette[category.split('\n')[1]] for category in category_proportion.index]

total_categories = len(category_proportion)
plt.figure(figsize=(18, 14))

# Calculate entropy for each category
category_entropy = {}

for category in desired_order:
    cluster_counts = category_proportion.loc[category].values
    cluster_probabilities = cluster_counts / cluster_counts.sum()
    category_entropy[category] = (-np.sum(cluster_probabilities * np.log(cluster_probabilities + 1e-9))) / np.log(len(cluster_probabilities))
category_proportion = category_proportion.reindex(desired_order)
# Define colors for the categories based on condition_palette
category_colors = [condition_palette[category.split('\n')[1]] for category in category_proportion.index]

for i, category in enumerate(desired_order):
    make_spider_for_category(category_proportion, category, category_proportion.columns, color=category_colors[i], subplot_index=i, total_subplots=total_categories)
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'spider_charts_biolabels_{("_").join(single_exp)}.pdf'))
plt.show()



# Prepare data for plotting
entropy_df = pd.DataFrame.from_dict(category_entropy, orient='index', columns=['entropy'])
entropy_df.reset_index(inplace=True)
entropy_df.rename(columns={'index': 'category'}, inplace=True)
entropy_df['condition'] = entropy_df['category'].apply(lambda x: x.split('\n')[1])
entropy_df['mutation'] = entropy_df['category'].apply(lambda x: x.split('\n')[0])

# Plotting
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(x='mutation', y='entropy', hue='condition', data=entropy_df, palette=condition_palette, dodge=True)

# Set labels, title, and legend
ax.set_xlabel('Mutation')
ax.set_ylabel('Entropy')
ax.set_title('Category Entropy by Condition and Mutation')
ax.legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'category_entropy_{("_").join(single_exp)}.pdf'))
plt.show()


## Class proportions in clusters

In [ ]:
# Compute class proportions in clusters
class_proportion = annotations_df.groupby(['cluster', 'mut_cond']).size().unstack(fill_value=0)
class_proportion = class_proportion.div(class_proportion.sum(axis=1), axis=0)

# Compute cluster proportions in classes
cluster_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
cluster_proportion = cluster_proportion.div(cluster_proportion.sum(axis=1), axis=0)

# Plot clustermap for class proportions in clusters
sns.clustermap(class_proportion, cmap='Greys', figsize=(10, 10))
plt.title('Class Proportions in Clusters')
plt.show()

## Plot clustermap for cluster proportions in classes
#sns.clustermap(cluster_proportion, cmap='Greys', figsize=(10, 10), row_cluster=False)
#plt.title('Cluster Proportions in Classes')
#plt.show()

In [ ]:
# Sort the dataframe based on the ordered mut_cond
annotations_df = annotations_df.sort_values('mut_cond').reset_index(drop=True)

# Calculate the proportion of clusters for each mut_cond
cluster_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
cluster_proportion = cluster_proportion.div(cluster_proportion.sum(axis=1), axis=0)

# Plotting
fig, ax = plt.subplots(figsize=(12, 8))
cluster_proportion.plot(kind='barh', stacked=True, ax=ax, color=palette[:len(cluster_proportion.columns)])

# Set labels, title, and legend
ax.set_xlabel('Proportion')
ax.set_ylabel('Mutation and Condition')
ax.set_title('Proportion of Clusters for Each Mutation and Condition')
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'cluster_proportion_{("_").join(single_exp)}.pdf'))
plt.show()
plt.close()

In [ ]:
# Display proportion of cluster labels in mutation condition classes as a pie chart
fig, axes = plt.subplots(nrows=len(cluster_proportion.index), ncols=1, figsize=(7, 7 * len(cluster_proportion.index)))
if len(cluster_proportion.index) == 1:
    axes = [axes]
for ax, mut_cond in zip(axes, cluster_proportion.index):
    proportions = cluster_proportion.loc[mut_cond]
    labels = [f'Cluster {cl}' for cl in proportions.index]
    colors = palette[:len(proportions)]
    ax.pie(proportions, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
    ax.set_title(f'Cluster Proportions in {mut_cond}')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'cluster_proportions_piecharts_{("_").join(single_exp)}.pdf'))
plt.show()

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=(10, 8))
class_proportion.plot(kind='bar', stacked=True, ax=ax, colormap='rainbow')

# Set labels, title, and legend
ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion')
ax.set_title('Proportion of Classes in Each Cluster')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()
plt.close()

In [ ]:
import random

# Function to display images
def display_images(img_codes):
    images = [Image.open(img_code) for img_code in img_codes]
    fig, axes = plt.subplots(7, 7, figsize=(15, 15))
    for ax, img in zip(axes.flatten(), images):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'Cluster {cluster_label}', fontsize=20)
    plt.tight_layout()
    plt.show()

# Select 49 random images for each cluster
for cluster_label in sorted(annotations_df['cluster'].unique()):
    cluster_images = annotations_df[annotations_df['cluster'] == cluster_label]['img_code'].tolist()
    if len(cluster_images) > 49:
        cluster_images = random.sample(cluster_images, 49)
    display_images(cluster_images)

In [ ]:
import random

# Function to display images
def display_images_row(img_codes, cluster_label, ax, adjust):
    images = [Image.open(img_code) for img_code in img_codes]
    for i, img in enumerate(images):
        ax[i].imshow(img)
        ax[i].axis('off')
    adjust = 0 if cluster_0 else 1
    ax[0].set_title(f'Cluster {cluster_label+adjust}', fontsize=20, color=cl_palette[cluster_label], fontweight='bold')

n_examples = 20

fig, axes = plt.subplots(nrows=len(annotations_df['cluster'].unique()), ncols=n_examples, figsize=(2*n_examples, 2 * len(annotations_df['cluster'].unique())))
for row, cluster_label in enumerate(sorted(annotations_df['cluster'].unique())):
    cluster_images = annotations_df[annotations_df['cluster'] == cluster_label]['img_code'].tolist()
    if len(cluster_images) > n_examples:
        cluster_images = random.sample(cluster_images, n_examples)
    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
    display_images_row(cluster_images, adjusted_cluster_label, axes[row], adjust=cluster_0)

plt.tight_layout()
plt.show()